# 04 — Step 3: Temporal Profiling

**Goal**: Plot CRA across 20 denoising timesteps to identify when reflection attention peaks.

In [ ]:
import sys
sys.path.insert(0, '/content/mi-mirror')

import pickle
import matplotlib.pyplot as plt
import numpy as np

from scripts.config import EXPERIMENTS_DIR, NUM_INFERENCE_STEPS
from scripts.metrics import compute_temporal_profiles, identify_peak_timestep
from scripts.visualization import plot_temporal_profiles

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP3_DIR = EXPERIMENTS_DIR / "step3_temporal"
STEP3_DIR.mkdir(parents=True, exist_ok=True)

with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)

candidates = step1["candidates"]
block_infos = step1["block_infos"]
mirror_data = step1["mirror_attention_data"]
nonmirror_data = step1["nonmirror_attention_data"]

print(f"Loaded {len(mirror_data)} mirror and {len(nonmirror_data)} non-mirror samples")
print(f"Top-5 candidates: {[(b, h) for b, h, _ in candidates[:5]]}")

In [ ]:
# Compute temporal CRA profiles
mirror_profiles = compute_temporal_profiles(
    mirror_data, candidates[:5], NUM_INFERENCE_STEPS
)
nonmirror_profiles = compute_temporal_profiles(
    nonmirror_data, candidates[:5], NUM_INFERENCE_STEPS
)

print("Mirror CRA Temporal Profiles (first & last timesteps):")
for (b, h), profile in mirror_profiles.items():
    print(f"  B{b}H{h}: t0={profile[0]:.4f}  t{len(profile)-1}={profile[-1]:.4f}  max={max(profile):.4f}")

In [ ]:
peak_t = identify_peak_timestep(mirror_profiles)
print(f"Peak timestep for mirror CRA: t={peak_t} / {NUM_INFERENCE_STEPS}")
frac = peak_t / NUM_INFERENCE_STEPS
if frac < 0.3:
    phase = "early (layout)"
elif frac < 0.7:
    phase = "mid (structure)"
else:
    phase = "late (detail)"
print(f"Denoising phase: {phase}")

In [ ]:
fig = plot_temporal_profiles(
    mirror_profiles, title="Temporal CRA Profile — Mirror Images",
)
fig.savefig(STEP3_DIR / "temporal_cra_mirror.png", dpi=150, bbox_inches="tight")
plt.show()

fig = plot_temporal_profiles(
    nonmirror_profiles, title="Temporal CRA Profile — Non-Mirror Images",
)
fig.savefig(STEP3_DIR / "temporal_cra_nonmirror.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
timesteps = list(range(NUM_INFERENCE_STEPS))

for (b, h), profile in mirror_profiles.items():
    axes[0].plot(timesteps, profile, marker='.', linewidth=2, label=f'B{b}H{h}')
axes[0].set_title('Mirror Images')
axes[0].set_xlabel('Denoising Step')
axes[0].set_ylabel('CRA')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for (b, h), profile in nonmirror_profiles.items():
    axes[1].plot(timesteps, profile, marker='.', linewidth=2, label=f'B{b}H{h}')
axes[1].set_title('Non-Mirror Images')
axes[1].set_xlabel('Denoising Step')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Temporal CRA Comparison', fontsize=14)
fig.tight_layout()
fig.savefig(STEP3_DIR / 'temporal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results = {
    "mirror_profiles": mirror_profiles,
    "nonmirror_profiles": nonmirror_profiles,
    "peak_timestep": peak_t,
}
with open(STEP3_DIR / "step3_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP3_DIR}")